# LDPC: classical to quantum

In [ ]:
import numpy as np
import stim

# Hypergraph product stabilizer matrices
def hypergraph_product(H1, H2):
    r1, n1 = H1.shape
    r2, n2 = H2.shape
    Ir1 = np.eye(r1, dtype=int)
    Ir2 = np.eye(r2, dtype=int)
    In1 = np.eye(n1, dtype=int)
    In2 = np.eye(n2, dtype=int)

    Hx = np.block([np.kron(H1, In2), np.kron(Ir1, H2.T)])
    Hz = np.block([np.kron(In1, H2), np.kron(H1.T, Ir2)])
    return Hx % 2, Hz % 2

## Generalized $H_x$

In [ ]:
# !pip install ldpc

In [ ]:
from ldpc.mod2 import nullspace, row_span, rank

def find_logical_x(Hx: np.ndarray, Hz: np.ndarray) -> np.ndarray:
    """
    Find a valid logical X operator: ker(Hx) \\ im(Hz.T)
    """
    assert Hx.shape[1] == Hz.shape[1], (
        f"Column mismatch: Hx has {Hx.shape[1]} cols, Hz has {Hz.shape[1]} cols"
    )
    
    kernel = nullspace(Hx).toarray().astype(int)
    rHz = rank(Hz)          

    for v in kernel:
        v = np.array(v).flatten()  # ensure 1D
        if v.shape[0] != Hz.shape[1]:
            # Pad or raise — should not happen if CSS is valid
            raise ValueError(f"v length {v.shape[0]} != Hz cols {Hz.shape[1]}")
        combined = np.vstack([Hz, v[np.newaxis, :]]) % 2
        if rank(combined) > rHz:
            return v

    raise ValueError("No logical X operator found!")


## Circuit funcs

In [ ]:
def css_x_memory(Hx: np.ndarray, p: float, Hz: np.ndarray) -> stim.Circuit:
    """
    Build a stim X-memory circuit from any binary parity-check matrix Hx.
    
    Hx:  (r x n) binary matrix — each row is one Z-stabilizer (detects X errors)
    p:   X error probability on data qubits
    Hz:  (optional) Z parity-check matrix — used to find true logical X operator
    """
    r, n = Hx.shape
    data = list(range(n))
    anc  = list(range(n, n + r))

    c = stim.Circuit()
    c.append("R", data + anc)
    c.append("X_ERROR", data, p)

    for i, row in enumerate(Hx):
        support = [j for j, val in enumerate(row) if val == 1]
        a = anc[i]
        c.append("H", a)
        for q in support:
            c.append("CZ", [a, q])
        c.append("H", a)
        c.append("M", a)
        c.append("DETECTOR", [stim.target_rec(-1)])

    c.append("M", data)

    # Use true logical X operator if Hz is provided, else fall back to Hx[0]
    logical_vec = find_logical_x(Hx, Hz)

    logical_support = [j for j, val in enumerate(logical_vec) if val == 1]
    c.append(
        "OBSERVABLE_INCLUDE",
        [stim.target_rec(-(n - q)) for q in sorted(logical_support)],
        0
    )
    return c

In [ ]:
import matplotlib.pyplot as plt
import os

In [ ]:
from ldpc.mod2 import rank
import numpy as np

def circulant_perm(L: int, s: int) -> np.ndarray:
    I = np.eye(L, dtype=int)
    return np.roll(I, s % L, axis=1)

def qc_parity_from_shifts(shift_mat: np.ndarray, L: int) -> np.ndarray:
    """
    shift_mat[i,j] = -1 => zero block, else circulant permutation with that shift.
    Returns binary parity-check matrix H over GF(2).
    """
    br, bc = shift_mat.shape
    blocks = []
    for i in range(br):
        row_blocks = []
        for j in range(bc):
            s = shift_mat[i, j]
            if s < 0:
                row_blocks.append(np.zeros((L, L), dtype=int))
            else:
                row_blocks.append(circulant_perm(L, int(s)))
        blocks.append(row_blocks)
    return np.block(blocks) % 2

def random_shift_matrix(mask: np.ndarray, L: int, rng: np.random.Generator) -> np.ndarray:
    S = np.full(mask.shape, -1, dtype=int)
    ones = np.argwhere(mask == 1)
    for i, j in ones:
        S[i, j] = int(rng.integers(0, L))
    return S

def build_hgp_for_lift(L: int, seed: int = 0, max_tries: int = 40, min_k: int = 1):
    """
    Build Hx,Hz using QC-LDPC base; retries random shifts until k >= min_k.
    """
    # Fixed sparse base mask (general for any L >= 2)
    base_mask = np.array([
        [1, 1, 0, 1, 0, 1],
        [0, 1, 1, 0, 1, 1],
        [1, 0, 1, 1, 1, 0],
    ], dtype=int)

    rng = np.random.default_rng(seed)
    for _ in range(max_tries):
        S = random_shift_matrix(base_mask, L, rng)
        H = qc_parity_from_shifts(S, L)      # rectangular LDPC possible
        Hx, Hz = hypergraph_product(H, H)    # CSS by construction

        k = Hx.shape[1] - rank(Hx) - rank(Hz)
        if k >= min_k:
            return H, Hx, Hz, k

    raise ValueError(f"Could not find code with k>={min_k} for L={L}")

def make_css_x_memory_experiment(n, p, seed=0):
    """
    n is now the QC lift factor (general size parameter).
    """
    _, Hx, Hz, k = build_hgp_for_lift(L=n, seed=seed, min_k=1)
    circ = css_x_memory(Hx, p=p, Hz=Hz)
    return sinter.Task(
        circuit=circ,
        json_metadata={"n": n, "p": p, "k": int(k), "n_physical": circ.num_qubits}
    )

## Execution

In [ ]:
def make_bp_osd_decoder(Hx: np.ndarray, p: float):
    """Create a BP+OSD decoder with compatibility fallbacks across ldpc versions."""
    H = Hx.astype(np.uint8)

    try:
        from ldpc import bposd_decoder
        return bposd_decoder(
            H,
            error_rate=float(p),
            max_iter=50,
            bp_method="product_sum",
            osd_order=0,
        )
    except Exception:
        pass

    try:
        from ldpc.bposd_decoder import BpOsdDecoder
        return BpOsdDecoder(
            H,
            error_rate=float(p),
            max_iter=50,
            bp_method="product_sum",
            osd_order=0,
        )
    except Exception as exc:
        raise ImportError(
            "Could not initialize BP+OSD decoder from ldpc. "
            "Try upgrading ldpc or checking available decoder constructors."
        ) from exc

def run_bp_osd_x_memory(Hx: np.ndarray, Hz: np.ndarray, p: float, shots: int, seed: int = 0):
    """
    Monte Carlo X-memory simulation using BP+OSD on syndrome s = Hx @ e (mod 2).
    Returns a dict compatible with the plotting cell.
    """
    rng = np.random.default_rng(seed)
    n = Hx.shape[1]
    logical_vec = find_logical_x(Hx, Hz).astype(np.uint8)

    decoder = make_bp_osd_decoder(Hx=Hx, p=p)

    errors = 0
    for _ in range(shots):
        e = (rng.random(n) < p).astype(np.uint8)
        s = (Hx @ e) % 2
        e_hat = np.array(decoder.decode(s.astype(np.uint8))).astype(np.uint8).reshape(-1) % 2
        if e_hat.shape[0] != n:
            raise ValueError(f"Decoder output length {e_hat.shape[0]} does not match n={n}")
        residual = (e ^ e_hat) % 2
        if int((logical_vec @ residual) % 2) == 1:
            errors += 1

    return {
        "shots": int(shots),
        "errors": int(errors),
        "logical_error_rate": float(errors / shots if shots > 0 else 0.0),
    }

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
import time

sizes = [3, 5, 7, 9]
error_rates = [0.01, 0.02, 0.03, 0.1]
shots_per_point = 10_000

# Keep worker count conservative to avoid oversubscription with ldpc internals.
max_workers = min(len(sizes) * len(error_rates), max(1, (os.cpu_count() or 2) - 1))

def _run_point_process(job):
    n, p, shots = job
    try:
        _, Hx, Hz, k = build_hgp_for_lift(L=int(n), seed=1234 + int(n), min_k=1)
        stats = run_bp_osd_x_memory(
            Hx=Hx,
            Hz=Hz,
            p=float(p),
            shots=int(shots),
            seed=1000 + int(1000 * float(p)) + int(n),
        )
        row = {
            "n": int(n),
            "p": float(p),
            "k": int(k),
            "n_physical": int(Hx.shape[1]),
            **stats,
        }
        return ("ok", row, None)
    except Exception as e:
        return ("err", {"n": int(n), "p": float(p)}, str(e))

t0 = time.perf_counter()
jobs = [(n, p, shots_per_point) for n in sizes for p in error_rates]

results = []
errors = []

with ProcessPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(_run_point_process, job) for job in jobs]
    for fut in as_completed(futures):
        status, payload, err = fut.result()
        if status == "ok":
            results.append(payload)
        else:
            errors.append((payload["n"], payload["p"], err))

results.sort(key=lambda r: (r["n"], r["p"]))
errors.sort(key=lambda t: (t[0], t[1]))

for row in results:
    print(f"n={row['n']}, p={row['p']:.3f} -> logical={row['logical_error_rate']:.4g} ({row['errors']}/{row['shots']})")

for n, p, why in errors:
    print(f"Skipping n={n}, p={p}: {why}")

elapsed = time.perf_counter() - t0
print("DONE! nResults =", len(results), f"workers={max_workers}", f"elapsed={elapsed:.1f}s")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for n in sizes[:-1]:
    pts = [r for r in results if r["n"] == n]
    pts.sort(key=lambda r: r["p"])
    xs = [r["p"] for r in pts]
    ys = [r["logical_error_rate"] for r in pts]
    if len(xs) > 0:
        ax.plot(xs, ys, marker="o", label=f"n={n}")

# Reference line: logical error rate equals physical error rate
ref_x = sorted(set(error_rates))
ax.plot(ref_x, ref_x, linestyle="--", color="black", linewidth=1.5, label="logical = physical")

ax.set_xlabel("Physical error rate p")
ax.set_ylabel("Logical error rate")
ax.set_title(f"Hypergraph-product LDPC codes: X-memory with BP+OSD [shots={shots_per_point}]")
ax.legend()
ax.set_xscale("log")
ax.set_yscale("log")
plt.tight_layout()
plt.show()